In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import os

In [2]:
df = pd.read_csv("loan_approval_data.csv")
print("Data loaded:", df.shape)


Data loaded: (1000, 20)


In [3]:
if "Applicant_ID" in df.columns:
    df = df.drop("Applicant_ID", axis=1)


In [4]:
categorical_cols = df.select_dtypes(["object"]).columns.tolist()
numerical_cols   = df.select_dtypes(["number"]).columns.tolist()

num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

In [5]:
le_edu    = LabelEncoder()
le_target = LabelEncoder()

df["Education_Level"] = le_edu.fit_transform(df["Education_Level"])
df["Loan_Approved"]   = le_target.fit_transform(df["Loan_Approved"])

In [6]:
ohe_cols = ["Employment_Status", "Marital_Status", "Loan_Purpose",
            "Property_Area", "Gender", "Employer_Category"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded     = ohe.fit_transform(df[ohe_cols])
encoded_df  = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(ohe_cols), index=df.index)
df = pd.concat([df.drop(columns=ohe_cols), encoded_df], axis=1)

In [7]:
df["DTI_Ratio_sq"]    = df["DTI_Ratio"]    ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2


In [8]:
X = df.drop(columns=["Loan_Approved", "DTI_Ratio", "Credit_Score"])
y = df["Loan_Approved"]

In [9]:
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"\nModel Performance:")
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score : {f1_score(y_test, y_pred):.4f}")



Model Performance:
  Accuracy : 0.8750
  Precision: 0.7903
  Recall   : 0.8033
  F1-Score : 0.7967


In [11]:
os.makedirs("artifacts", exist_ok=True)

with open("artifacts/model.pkl",         "wb") as f: pickle.dump(model,         f)
with open("artifacts/scaler.pkl",        "wb") as f: pickle.dump(scaler,        f)
with open("artifacts/ohe.pkl",           "wb") as f: pickle.dump(ohe,           f)
with open("artifacts/le_edu.pkl",        "wb") as f: pickle.dump(le_edu,        f)
with open("artifacts/le_target.pkl",     "wb") as f: pickle.dump(le_target,     f)
with open("artifacts/num_imp.pkl",       "wb") as f: pickle.dump(num_imp,       f)
with open("artifacts/feature_names.pkl", "wb") as f: pickle.dump(feature_names, f)

In [12]:
cat_options = {
    "Employment_Status": sorted(df.get("Employment_Status", pd.Series()).unique().tolist())
                          if "Employment_Status" in df.columns else [],
}
# Re-read raw CSV to get original category lists for dropdowns
raw = pd.read_csv("loan_approval_data.csv")
dropdown_options = {}
for col in ohe_cols + ["Education_Level"]:
    if col in raw.columns:
        dropdown_options[col] = sorted(raw[col].dropna().unique().tolist())

with open("artifacts/dropdown_options.pkl", "wb") as f:
    pickle.dump(dropdown_options, f)

print("\n✅ All artifacts saved to ./artifacts/")
print("   Run: streamlit run app.py")



✅ All artifacts saved to ./artifacts/
   Run: streamlit run app.py
